# 05 — sector / lap-time baseline

Wk2 Day 3. Goal: a moving-average baseline for next-lap pace, scored end-to-end on cached races. The number it puts on the board is the **floor** Phase 3 models must beat — so build it before any ML lands, and write the scoring harness before the model.

## Problem statement (no code yet — write this first, on purpose)

**What is the baseline trying to predict?**
Lap N's `LapTimeS` (lap time in seconds) for a given driver in a given race.

**What inputs does it have?**
Laps N−3, N−2, N−1 of the same driver, *same stint*, *same race*. Concretely: `LapTimeS` values from those three prior laps, averaged, used as the point prediction for lap N.

**What does it NOT see?**
- Lap N's own `LapTimeS` (the thing we're predicting).
- Any future laps (N+1, N+2, …).
- Any other driver's laps.
- Any other race's laps.
- Telemetry, tyre temps, weather, gap-to-leader, sector splits — none of it. This is the dumbest reasonable baseline by design.

**Train / test split.**
We do not train this baseline (it has no parameters — just a window size). The "test" is: for every lap N within a stint where N has ≥3 prior laps in that same stint, predict from those 3 laps and compare to the actual. The first `window` laps of every stint are unpredictable and are dropped (their prediction is `NaN`).

We hold the choice of `window` (1, 2, 3, 5, 7) as the only hyperparameter and pick it from the elbow of window-vs-MAE on the same 8-race pool. That's mildly self-serving — in Phase 3, the model will get a real train/test split per-race or leave-one-race-out. For this baseline, the floor it sets is on data it has *no* parameters to overfit to, so the choice of window across all 8 races is fine.

**Success metric.**
MAE (mean absolute error) in seconds. Reported both as an overall number across all (driver, stint, lap) triples with a non-NaN prediction, **and** per-race so we can see which circuits are easier / harder to predict with a flat lap-pace model.

**Why MAE and not RMSE?**
A 0.2 s miss and a 2.0 s miss should both contribute linearly. RMSE punishes outliers — useful later when SC/VSC laps and pit laps are involved, but for the leak-free in-stint baseline, MAE is what matches the question "how close, on average, is the prediction to the truth?"

## What "leak-free" means here, concretely

The killer bug to avoid is: the rolling mean for lap N accidentally includes lap N itself, or includes laps from the *next* stint (post-pit), or includes laps from the *previous* driver in a sorted frame.

Defences, all of which must hold:
1. Sort by `[Driver, StintId, LapNumber]` before any rolling op.
2. `groupby(['Driver', 'StintId'])` so the rolling window can never bleed across stint boundaries (post-pit) or across drivers.
3. `.shift(1)` *before* `.rolling(window).mean()` so lap N's prediction is built from laps strictly earlier than N.
4. After all that, the first `window` laps of every stint must be `NaN` in the prediction column. If `preds.isna().sum()` doesn't equal `n_stints * window` (roughly), something leaked.

## Acceptance bar for end of day

- A function `moving_average_baseline(laps_df, window=3) -> pd.Series` runs on Bahrain 2024 and prints an MAE.
- A reusable scoring module at `src/aris/eval/scoring.py` (`mae`, `rmse`, `per_race_mae`) imported by this notebook, with tests under `tests/test_scoring.py` passing.
- Multi-race MAE printed and written to `results/wk2-baseline-mae.csv` for all 8 cached races.
- Window sweep over `[1, 2, 3, 5, 7]` plotted; chosen window written into the CSV with the line `BASELINE: window=X, MAE=Y.YY s, 8 races, leakage-free per-stint shift`.


In [ ]:
from pathlib import Path

import fastf1
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CACHE = Path("..") / "fastf1_cache"
CACHE.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE))

print("pandas", pd.__version__, "| numpy", np.__version__, "| fastf1", fastf1.__version__)

## Block 2 — single-race baseline on Bahrain 2024

Stint detection from Day 2's notebook is reproduced inline below. Day 4 lifts it into `src/aris/physics/stint.py`; for now it lives here so the notebook is self-contained.

In [ ]:
def detect_stints(laps_df: pd.DataFrame) -> pd.DataFrame:
    """Enrich a laps frame with LapTimeS, CompoundChange, StintId (per-driver, 1-indexed)."""
    df = laps_df.sort_values(["Driver", "LapNumber"]).copy()
    df["LapTimeS"] = df["LapTime"].dt.total_seconds()
    df["CompoundChange"] = df.groupby("Driver")["Compound"].transform(lambda s: s != s.shift(1))
    df["StintId"] = df.groupby("Driver")["CompoundChange"].cumsum()
    return df


def filter_clean_laps(enriched: pd.DataFrame) -> pd.DataFrame:
    """Drop laps that don't represent steady-state pace: out-laps, in-laps, NaN times."""
    e = enriched.copy()
    e = e[e["LapTimeS"].notna()]
    e = e[e["PitOutTime"].isna() & e["PitInTime"].isna()]
    return e


def moving_average_baseline(laps_df: pd.DataFrame, window: int = 3) -> pd.Series:
    """Predict each lap's LapTimeS from a rolling mean of the prior `window` laps in the same stint.

    Output is aligned to laps_df's index. First `window` laps of every (Driver, StintId) are NaN
    because there are no prior in-stint laps to average.
    """
    return (
        laps_df
        .sort_values(["Driver", "StintId", "LapNumber"])
        .groupby(["Driver", "StintId"], sort=False)["LapTimeS"]
        .transform(lambda s: s.shift(1).rolling(window).mean())
    )

In [ ]:
session = fastf1.get_session(2024, "Bahrain", "R")
session.load(laps=True, telemetry=False, weather=False, messages=False)

enriched = detect_stints(session.laps)
clean = filter_clean_laps(enriched)

preds = moving_average_baseline(clean, window=3).reindex(clean.index)

n_stints = clean.groupby(["Driver", "StintId"]).ngroups
expected_nan = n_stints * 3
actual_nan = int(preds.isna().sum())
print(f"stints in Bahrain 2024 R (post-filter): {n_stints}")
print(f"expected NaN preds (n_stints * window = {n_stints} * 3): {expected_nan}")
print(f"actual NaN preds: {actual_nan}")
assert actual_nan <= expected_nan, "more NaNs than expected — investigate leak"

mask = preds.notna()
y_true = clean.loc[mask, "LapTimeS"].to_numpy()
y_pred = preds.loc[mask].to_numpy()
single_race_mae = float(np.mean(np.abs(y_true - y_pred)))
print(f"\nBahrain 2024 — moving-average-of-3 baseline MAE: {single_race_mae:.3f} s on {len(y_true)} laps")